<a href="https://colab.research.google.com/github/VitorTardivo21/Redes-Neurais-e-IA-Aplicada/blob/main/03_modelo_classificacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Etapa 3 — Análise Exploratória, Tokenização e Preparação do Dataset
Projeto: Diário Oficial Inteligente de Avaré

Atividade 1 — EDA (distribuição de classes, comprimento, termos frequentes)
Atividade 2 — Tokenização com stopwords
Atividade 3 — Construção do vocabulário ->vocab.json
Atividade 4 — Codificação dos rótulos ->label_map.json
Atividade 5 — Dataset e DataLoader PyTorch
Atividade 6 — Comparação com PyTorch-NLP (torchnlp)
"""

import os
import re
import json
import sys
import pandas as pd
import matplotlib
matplotlib.use("Agg")   # renderiza sem janela (compatível com qualquer ambiente)
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.model_selection import train_test_split

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from src.dataset import tokenizar, codificar_texto, DiarioDataset, STOPWORDS

# ==========================================
# CAMINHOS
# ==========================================

BASE_DIR       = os.path.dirname(os.path.abspath(__file__))
CSV_AMOSTRA    = os.path.join(BASE_DIR, "data", "processed", "amostra_rotulada.csv")
PASTA_PROC     = os.path.join(BASE_DIR, "data", "processed")
VOCAB_JSON     = os.path.join(PASTA_PROC, "vocab.json")
LABEL_MAP_JSON = os.path.join(PASTA_PROC, "label_map.json")
DIST_PNG       = os.path.join(BASE_DIR, "docs", "distribuicao_classes.png")
HIST_PNG       = os.path.join(BASE_DIR, "docs", "distribuicao_comprimento.png")

# ==========================================
# CARREGA OS DADOS
# ==========================================

print("=" * 55)
print("ETAPA 3 — Preparação para NLP")
print("=" * 55)

df = pd.read_csv(CSV_AMOSTRA, encoding="utf-8-sig")
print(f"\nAmostra carregada: {len(df)} registros | {df['rotulo'].nunique()} classes\n")

# ==========================================
# ATIVIDADE 1 — EDA
# ==========================================

print("-" * 55)
print("ATIVIDADE 1 — Análise Exploratória (EDA)")
print("-" * 55)

# Passo 1: Distribuição das classes
contagem = df["rotulo"].value_counts()
print("\nDistribuição de publicações por classe:")
print(contagem.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
contagem.plot(kind="bar", color="#2E75B6", ax=ax)
ax.set_title("Distribuição de publicações por classe", fontsize=13)
ax.set_xlabel("Classe")
ax.set_ylabel("Quantidade")
ax.tick_params(axis="x", rotation=25)
for p in ax.patches:
    ax.annotate(str(int(p.get_height())),
                (p.get_x() + p.get_width() / 2, p.get_height() + 0.1),
                ha="center", fontsize=10)
plt.tight_layout()
plt.savefig(DIST_PNG, dpi=120)
plt.close()
print(f"\nGráfico salvo: {DIST_PNG}")

# Passo 2: Comprimento dos textos (número de palavras)
df["n_palavras"] = df["texto"].str.split().apply(len)
print("\nComprimento dos textos (palavras):")
print(df["n_palavras"].describe().round(1).to_string())

percentil_90 = int(df["n_palavras"].quantile(0.90))
print(f"\nPercentil 90: {percentil_90} palavras ->usado como MAX_LEN")

fig, ax = plt.subplots(figsize=(8, 4))
df["n_palavras"].hist(bins=20, color="#1F4E79", ax=ax)
ax.axvline(percentil_90, color="red", linestyle="--", label=f"P90 = {percentil_90}")
ax.set_title("Distribuição do comprimento dos textos (nº de palavras)", fontsize=13)
ax.set_xlabel("Número de palavras")
ax.set_ylabel("Frequência")
ax.legend()
plt.tight_layout()
plt.savefig(HIST_PNG, dpi=120)
plt.close()
print(f"Gráfico salvo: {HIST_PNG}")

# Passo 3: Termos mais frequentes
print("\nTop 20 termos mais frequentes na base:")
todas_palavras = " ".join(df["texto"]).lower().split()
top20 = Counter(todas_palavras).most_common(20)
for termo, freq in top20:
    print(f"  {termo:<20} {freq:>5}x")

print("\nTop 10 termos por classe:")
for classe in sorted(df["rotulo"].unique()):
    textos = df[df["rotulo"] == classe]["texto"]
    palavras = " ".join(textos).lower().split()
    top10 = Counter(palavras).most_common(10)
    termos = ", ".join([f"{t}({n})" for t, n in top10])
    print(f"  [{classe}] {termos}")

# ==========================================
# ATIVIDADE 2 — TOKENIZAÇÃO
# ==========================================

print("\n" + "-" * 55)
print("ATIVIDADE 2 — Tokenização")
print("-" * 55)

exemplo = "Decreto Municipal nº 4.521 - Fica decretado ponto facultativo."
tokens_ex = tokenizar(exemplo)
print(f"\nExemplo de tokenização:")
print(f"  Entrada : {exemplo}")
print(f"  Saída   : {tokens_ex}")
print(f"  Stopwords usadas: {len(STOPWORDS)}")

# ==========================================
# ATIVIDADE 3 — VOCABULÁRIO
# ==========================================

print("\n" + "-" * 55)
print("ATIVIDADE 3 — Construção do Vocabulário")
print("-" * 55)

contador = Counter()
for texto in df["texto"]:
    contador.update(tokenizar(texto))

vocab = {"<PAD>": 0, "<UNK>": 1}
tokens_unicos   = 0
tokens_filtrados = 0

for token, freq in contador.most_common():
    if freq >= 2:
        vocab[token] = len(vocab)
        tokens_unicos += 1
    else:
        tokens_filtrados += 1

with open(VOCAB_JSON, "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

print(f"\nTokens únicos encontrados  : {tokens_unicos + tokens_filtrados}")
print(f"Filtrados (freq < 2)       : {tokens_filtrados}")
print(f"Tamanho final do vocabulário: {len(vocab)} (incluindo <PAD> e <UNK>)")
print(f"Arquivo salvo: {VOCAB_JSON}")

# ==========================================
# ATIVIDADE 4 — CODIFICAÇÃO DOS RÓTULOS
# ==========================================

print("\n" + "-" * 55)
print("ATIVIDADE 4 — Codificação dos Rótulos")
print("-" * 55)

classes   = sorted(df["rotulo"].unique())
label2id  = {classe: i for i, classe in enumerate(classes)}
id2label  = {i: classe for classe, i in label2id.items()}

with open(LABEL_MAP_JSON, "w", encoding="utf-8") as f:
    json.dump(label2id, f, ensure_ascii=False, indent=2)

df["rotulo_id"] = df["rotulo"].map(label2id)

print(f"\nMapeamento rótulo ->ID:")
for classe, idx in label2id.items():
    print(f"  {idx}  -> {classe}")
print(f"Arquivo salvo: {LABEL_MAP_JSON}")

# ==========================================
# ATIVIDADE 5 — DATASET PYTORCH
# ==========================================

print("\n" + "-" * 55)
print("ATIVIDADE 5 — Dataset e DataLoader PyTorch")
print("-" * 55)

try:
    import torch
    from torch.utils.data import DataLoader

    MAX_LEN = percentil_90  # valor calculado na EDA (P90 das palavras)

    # Divisão treino/teste estratificada por rótulo
    treino_df, teste_df = train_test_split(
        df, test_size=0.2, random_state=42, stratify=df["rotulo_id"]
    )
    print(f"\nDivisão: {len(treino_df)} treino | {len(teste_df)} teste (80/20 estratificado)")

    treino_ds = DiarioDataset(treino_df, vocab, label2id, max_len=MAX_LEN)
    teste_ds  = DiarioDataset(teste_df,  vocab, label2id, max_len=MAX_LEN)

    treino_loader = DataLoader(treino_ds, batch_size=8, shuffle=True)
    teste_loader  = DataLoader(teste_ds,  batch_size=8, shuffle=False)

    # Verifica o primeiro lote
    x_lote, y_lote = next(iter(treino_loader))
    print(f"\nFormato X (texto codificado): {tuple(x_lote.shape)}  ->[batch_size, MAX_LEN]")
    print(f"Formato y (rótulo)          : {tuple(y_lote.shape)}  ->[batch_size]")
    print(f"Rótulos do lote             : {y_lote.tolist()}")
    print(f"Classes do lote             : {[id2label[i] for i in y_lote.tolist()]}")
    print(f"MAX_LEN usado               : {MAX_LEN}")
    print(f"Valores de X em [0, {len(vocab)-1}]: {x_lote.min().item()} a {x_lote.max().item()}")
    print("\nDataset PyTorch OK.")

except ImportError as e:
    MAX_LEN = percentil_90
    print(f"PyTorch nao instalado: {e}")

# ==========================================
# ATIVIDADE 6 — PYTORCH-NLP (torchnlp)
# ==========================================

print("\n" + "-" * 55)
print("ATIVIDADE 6 — Comparação com PyTorch-NLP")
print("-" * 55)

try:
    from torchnlp.encoders.text import WhitespaceEncoder
    from torchnlp.encoders import LabelEncoder

    # Encoder de texto: constrói vocabulário e codifica automaticamente
    textos_tokenizados = df["texto"].apply(
        lambda t: " ".join(tokenizar(t))
    ).tolist()

    encoder_texto = WhitespaceEncoder(textos_tokenizados)
    exemplo_enc   = encoder_texto.encode("pregao eletronico registro precos")

    print(f"\nWhitespaceEncoder (torchnlp):")
    print(f"  Tamanho do vocabulário : {encoder_texto.vocab_size}")
    print(f"  Codificação de exemplo : {exemplo_enc[:10].tolist()}...")

    # Encoder de rótulos
    encoder_rotulo = LabelEncoder(df["rotulo"].tolist(), reserved_labels=[])
    ex_rotulo = encoder_rotulo.encode("decreto")
    print(f"\nLabelEncoder (torchnlp):")
    print(f"  'decreto' codificado como: {ex_rotulo}")

    print("\nComparação com implementação manual:")
    print(f"  Vocab manual  : {len(vocab)} tokens (freq >= 2 + PAD/UNK)")
    print(f"  Vocab torchnlp: {encoder_texto.vocab_size} tokens (sem filtro de frequência)")
    print(f"  Diferença     : {encoder_texto.vocab_size - len(vocab)} tokens (ruído e hápax)")

except ImportError as e:
    print(f"PyTorch-NLP nao instalado: {e}")
except Exception as e:
    print(f"Erro no torchnlp: {e}")

# ==========================================
# RELATÓRIO FINAL
# ==========================================

print("\n" + "=" * 55)
print("ETAPA 3 — CONCLUÍDA")
print("=" * 55)
print(f"Vocabulário     : {len(vocab)} tokens  -> {VOCAB_JSON}")
print(f"Label map       : {len(label2id)} classes -> {LABEL_MAP_JSON}")
print(f"MAX_LEN (P90)   : {MAX_LEN} palavras")
print(f"Gráficos        : docs/distribuicao_classes.png")
print(f"                  docs/distribuicao_comprimento.png")
